In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

NPZ_PATHS = ["../data/chess_data_snipped.npz"]
BATCH_SIZE = 256
VAL_FRAC   = 0.1
CKPT_PATH  = "../weights/

in_ch = 17

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
#-----------------------------
# Combine + create data set funcs/classes

import numpy as np
import torch
from torch.utils.data import Dataset, ConcatDataset

class ChessEvalNPZ(Dataset):
    def __init__(self, npz_path: str, flatten: bool = True, max_abs_cp: float = 13000.0):
        self.path = npz_path
        self.flatten = flatten
        with np.load(self.path, mmap_mode='r') as zf:
            Zcp = zf['evaluations']
            self.N = int(Zcp.shape[0])
            self.M_cp = float(max_abs_cp)
        self._X = None; self._Zcp = None

    def _ensure_open(self):
        if self._X is None:
            zf = np.load(self.path, mmap_mode='r')
            self._X   = zf['positions']      # 
            self._Zcp = zf['evaluations']    # (N,)        memmap

    def __len__(self): 
        return self.N

    def __getitem__(self, idx):
        self._ensure_open()
        x = self._X[idx].astype(np.float32)        
        if self.flatten:
            x = x.reshape(-1)
        z_cp = float(self._Zcp[idx])
        #v = 0.5*(z/M + 1),  clip
        # v = 0.5 * (z_cp / self.M_cp + 1.0)
        # v = np.clip(v, 0.0, 1.0)
        cp = np.clip(z_cp, -self.M_cp, self.M_cp)
        return torch.from_numpy(x), torch.tensor([cp], dtype=torch.float32)  #torch.from_numpy(x), torch.tensor([v], dtype=torch.float32)




In [3]:
# -----------------------------
# Building loaders



global_M = 1500.0
datasets = [ChessEvalNPZ(p, flatten=False, max_abs_cp=global_M) for p in NPZ_PATHS]
full_ds = ConcatDataset(datasets)
N = len(full_ds)
val_len = int(N * VAL_FRAC)
train_len = N - val_len

g = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full_ds, [train_len, val_len], generator=g)

NUM_WORKERS = 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Dataset: N={N} | input_dim={in_ch} | train={train_len} | val={val_len}")
xb0, yb0 = next(iter(train_loader))
print("One sample check:", xb0.shape)  # (B, 796)

Dataset: N=16057671 | input_dim=17 | train=14451904 | val=1605767
One sample check: torch.Size([256, 17, 8, 8])


In [4]:
xb0, yb0 = next(iter(train_loader))
print("One sample check:", xb0.shape)
assert xb0.shape[1] == in_ch, f"Expected {in_ch} channels, got {xb0.shape[1]}"


One sample check: torch.Size([256, 17, 8, 8])


In [5]:
# a) shapes & dtypes
print("xb0:", xb0.shape, xb0.dtype, "yb0:", yb0.shape, yb0.dtype)

# b) value ranges
xb_min, xb_max = float(xb0.min()), float(xb0.max())
print(f"X range first batch: [{xb_min}, {xb_max}]")

# c) targets in [0,1] after transform?
print("y batch stats: min=", float(yb0.min()), "max=", float(yb0.max()))

# d) no NaNs/inf in a couple random batches
def has_bad(t): 
    return torch.isnan(t).any().item() or torch.isinf(t).any().item()

for i, (xchk, ychk) in enumerate(train_loader):
    if i == 3: break
    assert not has_bad(xchk), "NaN/Inf in inputs"
    assert not has_bad(ychk), "NaN/Inf in targets"
print("Basic data checks passed.")


xb0: torch.Size([256, 17, 8, 8]) torch.float32 yb0: torch.Size([256, 1]) torch.float32
X range first batch: [-1.0, 1.0]
y batch stats: min= -1500.0 max= 1500.0
Basic data checks passed.


In [6]:
#More config
import random, numpy as np, torch

EPOCHS     = 100
PATIENCE   = 8

# CONV_CHANNELS = (192, 192, 192, 192)
# FC_HIDDEN = (256,)
# P_DROP = 0.1
CONV_CHANNELS = (128, 128, 128)
FC_HIDDEN     = (128,)
P_DROP        = 0.05

LR = 6e-4
weight_decay = 2e-4
betas = (0.9, 0.99)



SEED = 42


random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True



In [7]:
class CNN(nn.Module):
    def __init__(self, in_ch: int, conv_channels=(128,128,128), fc_hidden=(256,), p_drop=0.0):
        super().__init__()
        conv = []
        prev = in_ch
        for c in conv_channels:
            conv += [nn.Conv2d(prev, c, kernel_size=3, padding=1), nn.ReLU(inplace=True)]
            prev = c
        self.conv = nn.Sequential(*conv)

        self.pool = nn.AdaptiveAvgPool2d(1)  # -> (B,C,1,1)

        head = []
        prev = conv_channels[-1]  # NOT *8*8 anymore
        for h in fc_hidden:
            head += [nn.Linear(prev, h), nn.ReLU(inplace=True)]
            if p_drop and p_drop > 0:
                head += [nn.Dropout(p_drop)]
            prev = h
        head += [nn.Linear(prev, 1)]
        self.head = nn.Sequential(*head)

    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x).flatten(1)  # (B,C)
        return self.head(x)


In [8]:
import torch.nn as nn
import torch.nn.functional as F

class CenterWeightedHuber(nn.Module):
    def __init__(self, delta=150.0, clip=1500.0, k=400.0, min_w=0.5):
        super().__init__()
        self.delta = float(delta)
        self.clip  = float(clip)
        self.k     = float(k)
        self.min_w = float(min_w)

    def forward(self, pred_cp, true_cp):
        if pred_cp.dim() > 1 and pred_cp.size(-1) == 1:
            pred_cp = pred_cp.squeeze(-1)
        if true_cp.dim() > 1 and true_cp.size(-1) == 1:
            true_cp = true_cp.squeeze(-1)

        t = true_cp.clamp(-self.clip, self.clip)
        base = F.smooth_l1_loss(pred_cp, t, beta=self.delta, reduction='none')

        # emphasize near-equal positions
        w = torch.exp(- (t.abs()/self.k)**2)
        w = self.min_w + (1 - self.min_w) * w

        return (base * w).mean()



In [9]:
model = CNN(in_ch=in_ch, conv_channels=CONV_CHANNELS, fc_hidden=FC_HIDDEN, p_drop=P_DROP).to(device)



criterion = CenterWeightedHuber(delta=80, clip=1500, k=600, min_w=0.6) #CenterWeightedHuber(delta=150, clip=1500, k=800, min_w=0.8)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4,   # start here
    betas=(0.9, 0.99),
    eps=1e-8
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))


Nparams = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {Nparams/1e6:.3f}M")


Params: 0.332M


In [10]:
model.train()
xb, yb = xb0.to(device), yb0.to(device)
out = torch.zeros_like(model(xb))
print("forward:", out.shape)
loss = criterion(out, yb)
# loss.backward()
# torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
# optimizer.step(); optimizer.zero_grad(set_to_none=True)
print("one-step OK, loss:", float(loss))


forward: torch.Size([256, 1])
one-step OK, loss: 194.67861938476562


In [11]:
yb[0]

tensor([13.], device='cuda:0')

In [12]:
# ---- Clean tiny overfit test (no dropout, no fancy loss) ----

from torch.utils.data import Subset

small_idx = list(range(min(1024, len(train_ds))))
tiny_loader = DataLoader(
    Subset(train_ds, small_idx),
    batch_size=256,
    shuffle=True
)

# IMPORTANT: no dropout for overfit test
model_small = CNN(
    in_ch=in_ch,
    conv_channels=CONV_CHANNELS,
    fc_hidden=FC_HIDDEN,
    p_drop=0.0
).to(device)

# Lower LR, stable optimizer
opt_small = torch.optim.Adam(
    model_small.parameters(),
    lr=3e-4,
    betas=(0.9, 0.99),
    eps=1e-8
)

# Simple Huber (no center weighting)
crit = torch.nn.SmoothL1Loss(beta=150.0)

print("Starting tiny overfit test...")

for e in range(1500):   # 200–500 is enough
    tl = 0.0
    model_small.train()

    for xb, yb in tiny_loader:
        xb, yb = xb.to(device), yb.to(device)

        opt_small.zero_grad(set_to_none=True)
        pred = model_small(xb)
        loss = crit(pred, yb)

        loss.backward()
        opt_small.step()

        tl += loss.item() * xb.size(0)

    epoch_loss = tl / len(small_idx)
    print(f"(tiny) epoch {e+1:03d} loss {epoch_loss:.6f}")


Starting tiny overfit test...
(tiny) epoch 001 loss 236.090687
(tiny) epoch 002 loss 236.073078
(tiny) epoch 003 loss 236.035820
(tiny) epoch 004 loss 235.945099
(tiny) epoch 005 loss 235.759758
(tiny) epoch 006 loss 235.445068
(tiny) epoch 007 loss 234.926327
(tiny) epoch 008 loss 234.019310
(tiny) epoch 009 loss 233.014236
(tiny) epoch 010 loss 232.107353
(tiny) epoch 011 loss 231.596500
(tiny) epoch 012 loss 231.835255
(tiny) epoch 013 loss 231.465504
(tiny) epoch 014 loss 231.381248
(tiny) epoch 015 loss 231.395504
(tiny) epoch 016 loss 231.419632
(tiny) epoch 017 loss 231.393715
(tiny) epoch 018 loss 231.313175
(tiny) epoch 019 loss 231.298534
(tiny) epoch 020 loss 231.256660
(tiny) epoch 021 loss 231.240112
(tiny) epoch 022 loss 231.223083
(tiny) epoch 023 loss 231.209278
(tiny) epoch 024 loss 231.193188
(tiny) epoch 025 loss 231.200077
(tiny) epoch 026 loss 231.187401
(tiny) epoch 027 loss 231.161858
(tiny) epoch 028 loss 231.150784
(tiny) epoch 029 loss 231.110790
(tiny) epoch 

In [13]:
# -----------------------------
# Train loop + early stopping
# -----------------------------
best_val = float('inf')
pat = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            pred = model(xb)
            loss = criterion(pred, yb)

        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * xb.size(0)


    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            pred = model(xb)
            val_loss += criterion(pred, yb).item() * xb.size(0)

    val_loss /= len(val_loader.dataset)
    scheduler.step(val_loss)

    print(f"Epoch {epoch:02d} | train loss: {train_loss:.6f} | val loss: {val_loss:.6f}")

    if val_loss < best_val - 1e-6:
        best_val = val_loss
        pat = 0
        torch.save({
            "model_state": model.state_dict(),
            "input_dim": int(in_ch),
            "arch": list(FC_HIDDEN),
            "conv_channels": list(CONV_CHANNELS),
            "p_drop": float(P_DROP),
        }, CKPT_PATH)


    else:
        pat += 1
        if pat >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best val loss: {best_val:.6f}")
            break

print("Best val loss:", best_val)
print(f"Saved best model to: {CKPT_PATH}")

Epoch 01 | train loss: 161.149053 | val loss: 147.314160
Epoch 02 | train loss: 142.426064 | val loss: 136.265952
Epoch 03 | train loss: 133.395255 | val loss: 130.829091
Epoch 04 | train loss: 127.904738 | val loss: 128.188874
Epoch 05 | train loss: 124.300686 | val loss: 121.809017
Epoch 06 | train loss: 121.580470 | val loss: 119.819767
Epoch 07 | train loss: 119.299767 | val loss: 117.543294
Epoch 08 | train loss: 117.442588 | val loss: 115.807476
Epoch 09 | train loss: 115.913124 | val loss: 114.732879
Epoch 10 | train loss: 114.539177 | val loss: 113.741972
Epoch 11 | train loss: 113.401266 | val loss: 112.343283
Epoch 12 | train loss: 112.385535 | val loss: 112.247641
Epoch 13 | train loss: 111.512881 | val loss: 111.028729
Epoch 14 | train loss: 110.659733 | val loss: 111.175029
Epoch 15 | train loss: 109.953114 | val loss: 110.479742
Epoch 16 | train loss: 109.303359 | val loss: 110.986074
Epoch 17 | train loss: 108.731806 | val loss: 110.936699
Epoch 18 | train loss: 108.1645

In [ ]:
def evaluate_mse(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            pred = model(xb)
            total += criterion(pred, yb).item() * xb.size(0)
            n += xb.size(0)
    return total / n

final_val_mse = evaluate_mse(model, val_loader)
print("Final (reloaded) val MSE:", final_val_mse)

NameError: name 'model' is not defined